# 🗺️ Google Haritalar İşletme Verilerini Çeken Bot - Kapsamlı Rehber

Bu notebook, **Google Haritalar üzerinden işletme bilgilerini otomatik olarak çeken** botun tüm kodlarını, mantığını ve teknik detaylarını adım adım açıklamaktadır.

## 📸 Uygulama Ekran Görüntüsü

![Uygulama Ekran Görüntüsü](public/Ekran%20görüntüsü%202026-06-06%20210542.png)

*Uygulama "Pizza İstanbul" araması ile 20 işletmenin verilerini başarıyla çekmiştir.*

---

## 📋 İçindekiler

1. [Proje Tanımı ve Amaç](#1-proje-tanimi)
2. [Gerekli Kütüphaneler ve Kurulum](#2-kutuphaneler)
3. [Tkinter GUI Tasarımı](#3-gui-tasarimi)
4. [Selenium WebDriver Yapılandırması](#4-webdriver)
5. [Google Haritalar Navigasyonu](#5-navigasyon)
6. [Çerez Onay Ekranı Yönetimi](#6-cerez)
7. [Arama Kutusu ve Dinamik Selector Sistemi](#7-arama)
8. [İşletme Verilerini Çekme](#8-veri-cekme)
9. [Akıllı Kaydırma Mekanizması](#9-kaydirma)
10. [Thread-Safe UI Güncellemesi](#10-thread-safe)
11. [Excel'e Aktarma](#11-excel)
12. [WhatsApp Entegrasyonu](#12-whatsapp)
13. [Hata Yönetimi ve Debugging](#13-hata-yonetimi)
14. [Tam Kaynak Kod](#14-tam-kod)

---

## 1. Proje Tanımı ve Amaç <a id='1-proje-tanimi'></a>

### 🎯 Problem

Belirli bir sektördeki veya bölgedeki işletmelerin iletişim bilgilerine ulaşmak zaman alıcı bir süreçtir. Google Haritalar bu bilgileri barındırsa da, yüzlerce işletmeyi tek tek ziyaret edip bilgilerini not etmek pratik değildir.

### 💡 Çözüm

Bu bot, **Selenium WebDriver** kullanarak Google Haritalar'ı otomatik olarak kontrol eder:

1. Microsoft Edge tarayıcısını arka planda başlatır
2. Google Haritalar'a gider
3. Kullanıcının belirlediği terimi arar
4. Sonuçları tek tek tıklayarak detay bilgilerini toplar
5. Bilgileri GUI tablosuna ve Excel'e aktarır

### 🛠️ Kullanılan Teknolojiler

| Teknoloji | Kullanım Alanı |
|-----------|---------------|
| **Selenium** | Tarayıcı otomasyonu ve web scraping |
| **Tkinter** | Grafiksel kullanıcı arayüzü (GUI) |
| **Pandas** | Veri yapılandırma ve Excel aktarımı |
| **Threading** | Arka plan işlemleri (UI donmaması için) |
| **Regular Expressions** | Telefon numarası formatlama |

---

## 2. Gerekli Kütüphaneler ve Kurulum <a id='2-kutuphaneler'></a>

### Kurulum Komutu

```bash
pip install selenium pandas openpyxl
```

### Kütüphanelerin İçe Aktarılması

In [ ]:
# === TEMEL KÜTÜPHANELER ===

# GUI (Grafiksel Kullanıcı Arayüzü) için Tkinter
import tkinter as tk
from tkinter import ttk, filedialog

# Web otomasyon için Selenium
from selenium import webdriver                              # Tarayıcı kontrolü
from selenium.webdriver.common.by import By                 # Element bulma yöntemleri
from selenium.webdriver.common.keys import Keys             # Klavye tuşları (ENTER, TAB vb.)
from selenium.webdriver.common.action_chains import ActionChains  # Fare ve klavye aksiyonları
from selenium.webdriver.support.ui import WebDriverWait     # Akıllı bekleme
from selenium.webdriver.support import expected_conditions as EC  # Bekleme koşulları

# Yardımcı kütüphaneler
import time           # Bekleme süreleri
import threading      # Çoklu iş parçacığı (UI donmaması için)
import pandas as pd   # Veri işleme ve Excel aktarımı
import webbrowser     # WhatsApp Web bağlantısı açma
import re             # Düzenli ifadeler (telefon numarası formatlama)

print("✅ Tüm kütüphaneler başarıyla içe aktarıldı!")

### Kütüphane Açıklamaları

| Kütüphane | Modül | Açıklama |
|-----------|-------|----------|
| `tkinter` | `tk` | Python'un yerleşik GUI kütüphanesi |
| `tkinter.ttk` | `ttk` | Tema destekli Tkinter widget'ları (Treeview, Scrollbar) |
| `tkinter.filedialog` | `filedialog` | Dosya kaydetme/açma diyalogu |
| `selenium.webdriver` | `webdriver` | Chrome, Edge, Firefox tarayıcı sürücüleri |
| `selenium.webdriver.common.by` | `By` | `By.ID`, `By.CLASS_NAME`, `By.XPATH` gibi seçiciler |
| `selenium.webdriver.common.keys` | `Keys` | `Keys.ENTER`, `Keys.TAB` gibi tuşlar |
| `selenium.webdriver.support.ui` | `WebDriverWait` | Belirli bir koşul gerçekleşene kadar bekleme |
| `selenium.webdriver.support.expected_conditions` | `EC` | `presence_of_element_located`, `element_to_be_clickable` gibi koşullar |
| `pandas` | `pd` | DataFrame oluşturma ve Excel yazma |
| `re` | `re` | `re.sub()` ile metin temizleme |

---

## 3. Tkinter GUI Tasarımı <a id='3-gui-tasarimi'></a>

### GUI Bileşenleri

Uygulamamız 3 ana bölümden oluşur:

```
┌────────────────────────────────────────────┐
│  ÜSTTE: Arama Kutusu + İşletme Sayısı     │
│         [Verileri Çek] [Excel'e Aktar]      │
├────────────────────────────────────────────┤
│                                            │
│  ALTTA: Sonuç Tablosu (Treeview)           │
│  İşletme | Adres | Tel | Durum | WhatsApp  │
│                                            │
└────────────────────────────────────────────┘
```

In [ ]:
# === ANA PENCERE OLUŞTURMA ===
# Not: Bu hücre sadece gösterim amaçlıdır, çalıştırmak için isletme_verileri.py dosyasını kullanın.

class GoogleMapsApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Google Haritalar Arama Botu")  # Pencere başlığı
        self.root.geometry("1080x720")                   # Pencere boyutu
        self.root.configure(bg="#FFFFFF")                 # Arka plan rengi (beyaz)

        # === ÜST PANEL: Arama Kutusu ve Butonlar ===
        self.frame_top = tk.Frame(self.root, bg="#FFFFFF", bd=10, relief=tk.FLAT)
        self.frame_top.place(relx=0.02, rely=0.02, relwidth=0.96, relheight=0.2)

        # Arama kelimesi etiketi ve giriş kutusu
        self.label_search = tk.Label(
            self.frame_top, 
            text="Aramak İstediğiniz Kelime:", 
            bg="#FFFFFF", 
            font=("Helvetica", 10)
        )
        self.label_search.grid(row=0, column=0, padx=5, pady=5, sticky="w")
        
        self.entry_search = tk.Entry(
            self.frame_top, 
            font=("Helvetica", 10), 
            bd=5, 
            relief=tk.FLAT, 
            fg="black", 
            bg="#F8D8E8"  # Pembe arka plan
        )
        self.entry_search.grid(row=0, column=1, padx=5, pady=5)

        print("GUI yapısı gösterildi (çalıştırma için isletme_verileri.py kullanın)")

### Treeview Tablosu

Tkinter'ın `ttk.Treeview` widget'ı, çekilen verileri tablo formatında göstermek için kullanılır:

| Sütun | Açıklama | Genişlik |
|-------|----------|----------|
| İşletme Adı | İşletmenin Google Haritalar'daki adı | 150px |
| Adres | İşletmenin tam adresi | 250px |
| İletişim No | Telefon numarası (+90 formatında) | 120px |
| Mesaj Atıldı Mı? | WhatsApp mesajı gönderildi mi? | 100px |
| Mesaj Gönder | WhatsApp mesaj gönderme butonu | 100px |

In [ ]:
# === TREEVIEW TABLOSU OLUŞTURMA ===

# Treeview'ın sütun tanımları
columns = ("İşletme Adı", "Adres", "İletişim No", "Mesaj Atıldı Mı?", "Mesaj Gönder")

# Her sütunun yapılandırması
column_config = {
    "İşletme Adı": {"width": 150, "heading": "İşletme Adı"},
    "Adres":       {"width": 250, "heading": "Adres"},
    "İletişim No": {"width": 120, "heading": "İletişim No"},
    "Mesaj Atıldı Mı?": {"width": 100, "heading": "Mesaj Atıldı Mı?"},
    "Mesaj Gönder":     {"width": 100, "heading": "Mesaj Gönder"}
}

print("Treeview sütun yapılandırması:")
for col, config in column_config.items():
    print(f"  📊 {col}: genişlik={config['width']}px")

---

## 4. Selenium WebDriver Yapılandırması <a id='4-webdriver'></a>

### Neden Edge Tarayıcısı?

Bu projede **Microsoft Edge** varsayılan tarayıcı olarak kullanılmaktadır çünkü:

1. **Windows'ta yüklü gelir** - Ekstra kurulum gerektirmez
2. **Chromium tabanlıdır** - Chrome ile aynı motoru kullanır
3. **Selenium uyumludur** - `webdriver.Edge()` ile doğrudan kullanılabilir

### Anti-Bot Koruması

Google, otomatik tarayıcıları tespit edebilir. Bunu engellemek için özel ayarlar kullanırız:

In [ ]:
# === WEBDRIVER YAPILANDIRMASI ===

# Edge tarayıcı seçenekleri
options = webdriver.EdgeOptions()

# 1. Pencere boyutu ayarlama
options.add_argument("--window-size=1280,1024")
print("✅ Pencere boyutu: 1280x1024")

# 2. Otomasyon tespitini devre dışı bırakma
# Bu argüman, Chrome/Edge'in 'navigator.webdriver' özelliğini gizler
options.add_argument("--disable-blink-features=AutomationControlled")
print("✅ AutomationControlled devre dışı bırakıldı")

# 3. 'Chrome is being controlled by automated software' mesajını gizleme
options.add_experimental_option("excludeSwitches", ["enable-automation"])
print("✅ Otomasyon uyarısı gizlendi")

# 4. Otomasyon uzantısını devre dışı bırakma
options.add_experimental_option("useAutomationExtension", False)
print("✅ Otomasyon uzantısı devre dışı")

print("\n🔧 WebDriver yapılandırması tamamlandı!")
print("Not: driver = webdriver.Edge(options=options) ile tarayıcı başlatılır")

### Tarayıcı Fallback Sistemi

Eğer Edge başlatılamazsa, otomatik olarak Chrome'a geçilir:

```python
driver = None
try:
    driver = webdriver.Edge(options=edge_options)      # Önce Edge dene
except:
    try:
        driver = webdriver.Chrome(options=chrome_options)  # Edge yoksa Chrome dene
    except:
        return  # Her ikisi de yoksa işlemi sonlandır
```

Bu yapı sayesinde uygulama, kullanıcının bilgisayarında hangi tarayıcı yüklü olursa olsun çalışabilir.

---

## 5. Google Haritalar Navigasyonu <a id='5-navigasyon'></a>

### Google Haritalar Ana Sayfası

![Google Haritalar Ana Sayfa](public/google_maps_anasayfa.png)

*Selenium ile açılan Google Haritalar ana sayfası*

### Arama Sonuçları

![Google Haritalar Sonuçlar](public/google_maps_sonuclar.png)

*"eczane" araması sonucunda elde edilen işletme listesi*

---

## 6. Çerez Onay Ekranı Yönetimi <a id='6-cerez'></a>

Google, ilk ziyarette bir çerez onay ekranı gösterebilir. Bu ekranı otomatik olarak geçmemiz gerekir.

### Çerez Onayı Akış Şeması

```
Google Haritalar'ı Aç
        │
        ▼
  Çerez ekranı var mı?
   ╱          ╲
  Evet         Hayır
   │            │
   ▼            │
 iframe var mı? │
  ╱       ╲     │
 Evet     Hayır │
  │         │   │
  ▼         │   │
iframe'e    │   │
geç         │   │
  │         │   │
  └────┬────┘   │
       ▼        │
 "Kabul Et"     │
  butonuna tıkla│
       │        │
       ▼        │
  Ana içeriğe   │
  geri dön      │
       │        │
       └────────┘
            │
            ▼
    Arama kutusunu bul
```

In [ ]:
# === ÇEREZ ONAY EKRANI YÖNETİMİ ===

# Bu kod, Google'ın çerez onay ekranını otomatik olarak geçer.
# Çerez ekranı bazen bir iframe içinde olabilir.

cerez_yonetimi_kodu = '''
# 1. Aşama: iframe kontrolü
try:
    WebDriverWait(driver, 4).until(
        EC.frame_to_be_available_and_switch_to_it(
            (By.XPATH, "//iframe[contains(@src, 'consent.google') or contains(@title, 'Consent')]")
        )
    )
    print("Çerez onay iframe'ine geçiş yapıldı.")
except:
    pass  # iframe yoksa devam et

# 2. Aşama: Kabul et butonunu bul ve tıkla
accept_button = WebDriverWait(driver, 5).until(
    EC.element_to_be_clickable((By.XPATH, 
        "//button[contains(., 'Tümünü kabul et') or "
        "contains(., 'Accept all') or "
        "contains(., 'Kabul et') or "
        "contains(., 'I agree')]"
    ))
)
accept_button.click()

# 3. Aşama: iframe'den çık
driver.switch_to.default_content()
'''

print("Çerez onay yönetimi kodu:")
print(cerez_yonetimi_kodu)
print("\n💡 Bu kod Türkçe ve İngilizce çerez ekranlarını destekler.")

---

## 7. Arama Kutusu ve Dinamik Selector Sistemi <a id='7-arama'></a>

### Problem: Google'ın Değişen HTML Yapısı

Google, HTML element ID'lerini ve sınıf adlarını düzenli olarak değiştirir. Örneğin:

| Tarih | Arama Kutusu ID |
|-------|----------------|
| 2024 | `searchboxinput` |
| 2026 (Güncel) | `ucc-1` (dinamik) |

### Çözüm: Çoklu Selector Fallback Sistemi

Tek bir selector'a güvenmek yerine, birden fazla selector denenir:

In [ ]:
# === DİNAMİK SELECTOR SİSTEMİ ===

# Google'ın HTML yapısı değişse bile çalışacak selector listesi
selectors = [
    ("By.ID",          "searchboxinput",
     "Eski Google Maps ID - artık çoğu durumda çalışmaz"),
    
    ("By.XPATH",       "//form[contains(@jsaction, 'searchbox')]//input",
     "✅ EN GÜVENILIR: Form'un jsaction özniteliği üzerinden bulma"),
    
    ("By.XPATH",       "//input[contains(@class, 'UGojuc')]",
     "CSS sınıf adı ile bulma"),
    
    ("By.CSS_SELECTOR","input.UGojuc",
     "CSS seçici ile bulma (alternatif)"),
    
    ("By.XPATH",       "//input[@id='ucc-1']",
     "Dinamik ID ile bulma (son çare)")
]

print("🔍 Selector Öncelik Sırası:")
print("=" * 70)
for i, (method, selector, desc) in enumerate(selectors, 1):
    print(f"\n{i}. {method}: {selector}")
    print(f"   → {desc}")

print("\n" + "=" * 70)
print("\n💡 Her selector 5 saniye denenir. Bulunan ilk elemana yazma yapılır.")
print("   Hiçbiri bulunamazsa Exception fırlatılır.")

---

## 8. İşletme Verilerini Çekme <a id='8-veri-cekme'></a>

### Veri Çekme Akışı

Her işletme için şu adımlar izlenir:

```
1. Sonuç listesinde işletmeyi bul (By.CLASS_NAME, "Nv2PK")
       │
       ▼
2. İşletme kartına tıkla
       │
       ▼
3. Detay paneli açılır
       │
       ├──→ İşletme Adı:  h1.DUwDvf
       ├──→ Adres:        button[data-item-id='address'] .Io6YTe
       └──→ Telefon:      button[data-item-id^='phone'] .Io6YTe
       │
       ▼
4. Verileri tabloya ekle
       │
       ▼
5. Sonraki işletmeye geç
```

In [ ]:
# === CSS SELECTOR HARİTASI ===

# Google Haritalar'daki elementlerin selector haritası
google_maps_selectors = {
    "İşletme Kartı": {
        "selector": "By.CLASS_NAME, 'Nv2PK'",
        "aciklama": "Sonuç listesindeki her işletme kartı",
        "ornek": "<div class='Nv2PK'>Doğu Eczanesi</div>"
    },
    "İşletme Adı": {
        "selector": "By.XPATH, '//h1[contains(@class, 'DUwDvf')]'",
        "aciklama": "Detay sayfasındaki H1 başlık elementi",
        "ornek": "<h1 class='DUwDvf'>Pizza Hut</h1>"
    },
    "Adres": {
        "selector": "By.CSS_SELECTOR, 'button[data-item-id=address] .Io6YTe'",
        "aciklama": "Adres bilgisi butonu içindeki metin",
        "ornek": "Kadıköy, Moda Cd. No:12"
    },
    "Telefon": {
        "selector": "By.CSS_SELECTOR, 'button[data-item-id^=phone] .Io6YTe'",
        "aciklama": "Telefon butonu içindeki metin (^ ile başlayan eşleşme)",
        "ornek": "(0216) 371 03 50"
    }
}

print("🗺️ Google Haritalar Element Haritası")
print("=" * 60)
for element, bilgi in google_maps_selectors.items():
    print(f"\n📌 {element}")
    print(f"   Selector: {bilgi['selector']}")
    print(f"   Açıklama: {bilgi['aciklama']}")
    print(f"   Örnek:    {bilgi['ornek']}")

### Telefon Numarası Formatlama

Google Haritalar'dan gelen telefon numaraları farklı formatlarda olabilir. Bunları WhatsApp formatına çevirmemiz gerekir:

In [ ]:
# === TELEFON NUMARASI FORMATLAMA ===

import re

# Google Haritalar'dan gelebilecek farklı formatlar
ornek_numaralar = [
    "(0216) 371 03 50",
    "0216 371 03 50",
    "+90 216 371 0350",
    "02163710350",
    "(0546) 740 24 27",
]

print("📱 Telefon Numarası Formatlama:")
print("=" * 50)

for numara in ornek_numaralar:
    # 1. Adım: Sadece rakamları al
    temiz = re.sub(r'\D', '', numara)
    
    # 2. Adım: Son 10 haneyi al ve +90 ekle
    whatsapp_format = f'+90{temiz[-10:]}'
    
    print(f"  Orijinal: {numara:25s} → WhatsApp: {whatsapp_format}")

print("\n💡 re.sub(r'\\D', '', numara) tüm rakam olmayan karakterleri siler")
print("   temiz[-10:] son 10 haneyi alır (alan kodu dahil)")

---

## 9. Akıllı Kaydırma Mekanizması <a id='9-kaydirma'></a>

### Problem: Google Haritalar'ın Sonsuz Kaydırma Yapısı

Google Haritalar, sonuçları **sonsuz kaydırma (infinite scroll)** ile yükler. Ancak kaydırma tüm sayfada değil, sadece **sol sidebar panelde** gerçekleşir.

### Yanlış Yaklaşım ❌
```python
# Bu ÇALIŞMAZ çünkü Google Haritalar'da sayfa kaydırması yoktur
driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
```

### Doğru Yaklaşım ✅
```python
# Sonuç panelini (sidebar feed) hedefle ve onu kaydır
panel = driver.find_element(By.XPATH, "//div[@role='feed']")
driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", panel)
```

In [ ]:
# === AKILLI KAYDIRMA MEKANİZMASI ===

kaydirma_kodu = '''
# 1. Sonuç panelini bul
try:
    panel = driver.find_element(By.XPATH, "//div[@role='feed']")
except:
    try:
        panel = driver.find_element(
            By.CSS_SELECTOR, 
            "div[aria-label^='Results for'], div[aria-label^='Arama sonuçları']"
        )
    except:
        panel = None

if panel:
    # 2. Mevcut yüksekliği kaydet
    last_height = driver.execute_script(
        "return arguments[0].scrollHeight", panel
    )
    
    # 3. Paneli en alta kaydır
    driver.execute_script(
        "arguments[0].scrollTop = arguments[0].scrollHeight", panel
    )
    time.sleep(2)  # Yüklenmesini bekle
    
    # 4. Yeni yüksekliği kontrol et
    new_height = driver.execute_script(
        "return arguments[0].scrollHeight", panel
    )
    
    # 5. Değişim yoksa listenin sonuna gelindi
    if new_height == last_height:
        print("Listenin sonuna gelindi.")
        break
'''

print("🔄 Akıllı Kaydırma Mekanizması")
print("=" * 50)
print(kaydirma_kodu)
print("💡 scrollHeight değişmezse, yeni veri yüklenmemiş demektir.")

---

## 10. Thread-Safe UI Güncellemesi <a id='10-thread-safe'></a>

### Problem: "main thread is not in main loop" Hatası

Tkinter'ın GUI widget'ları **sadece ana thread'den** güncellenebilir. Selenium işlemleri ayrı bir thread'de çalıştığı için, doğrudan `self.tree.insert()` çağırmak hata verir.

### Çözüm: `_update_ui()` Helper Metodu

In [ ]:
# === THREAD-SAFE UI GÜNCELLEMESİ ===

# ❌ YANLIŞ: Arka plan thread'inden doğrudan widget güncellemesi
print("❌ YANLIŞ YAKLAŞIM:")
print("   self.tree.insert('', 'end', values=data)")
print("   → RuntimeError: main thread is not in main loop")

print()

# ✅ DOĞRU: root.after() ile ana thread'e yönlendirme
print("✅ DOĞRU YAKLAŞIM:")
print("   def _update_ui(self, func):")
print("       try:")
print("           self.root.after(0, func)")
print("       except RuntimeError:")
print("           pass")
print()
print("   # Kullanım:")
print("   row_data = (business_name, address, phone_number, 'Hayır', 'Mesaj Gönder')")
print("   self._update_ui(lambda d=row_data: self.tree.insert('', 'end', values=d))")

print()
print("💡 root.after(0, func) → func'ı ana thread'in event loop'unda çalıştırır")
print("   lambda d=row_data: ... → closure sorunu yaşamamak için default parametre kullanılır")

### Threading Mimarisi

```
┌─────────────────────────────────────────────────┐
│                 ANA THREAD                       │
│  (Tkinter Event Loop - root.mainloop())         │
│                                                  │
│  ┌──────────┐  ┌──────────┐  ┌──────────────┐  │
│  │ GUI      │  │ Event    │  │ _update_ui() │  │
│  │ Çizimi   │  │ Handler  │  │ Kuyruk       │  │
│  └──────────┘  └──────────┘  └──────┬───────┘  │
│                                      ▲          │
└──────────────────────────────────────┼──────────┘
                                       │
                            root.after(0, func)
                                       │
┌──────────────────────────────────────┼──────────┐
│              ARKA PLAN THREAD (daemon)│          │
│  (threading.Thread - scrape_data)     │          │
│                                       │          │
│  Selenium → Veri Çek → _update_ui()──┘          │
│                                                  │
└──────────────────────────────────────────────────┘
```

---

## 11. Excel'e Aktarma <a id='11-excel'></a>

In [ ]:
# === EXCEL'E AKTARMA ===

import pandas as pd

# Örnek veri (normalde Treeview'dan çekilir)
ornek_veri = [
    ("Arbor Hotel", "Küçük Çamlıca Cd. No:14", "+902166291009", "Hayır", "Mesaj Gönder"),
    ("ELK Pizza", "Şerifali, Miras Sk. No:7/b", "+902164558651", "Hayır", "Mesaj Gönder"),
    ("Pizza Panza", "Ataşehir Atatürk Mah.", "+902162666808", "Hayır", "Mesaj Gönder"),
]

# DataFrame oluştur
df = pd.DataFrame(
    ornek_veri, 
    columns=["İşletme Adı", "Adres", "İletişim No", "Mesaj Atıldı Mı?", "Mesaj Gönder"]
)

print("📊 DataFrame Önizleme:")
print(df.to_string(index=False))

print("\n💡 df.to_excel('dosya.xlsx', index=False) ile Excel'e aktarılır.")
print("   openpyxl kütüphanesi gereklidir: pip install openpyxl")

---

## 12. WhatsApp Entegrasyonu <a id='12-whatsapp'></a>

In [ ]:
# === WHATSAPP ENTEGRASYONU ===

import webbrowser

# WhatsApp Web API kullanımı
telefon_numarasi = "+902166291009"
whatsapp_url = f"https://wa.me/{telefon_numarasi}"

print("📱 WhatsApp Entegrasyonu")
print("=" * 50)
print(f"\n  Telefon: {telefon_numarasi}")
print(f"  URL:     {whatsapp_url}")
print(f"\n  Kullanım: webbrowser.open('{whatsapp_url}')")
print("\n💡 Bu URL, WhatsApp Web veya masaüstü uygulamasını açar.")
print("   Kullanıcı mesaj yazdıktan sonra gönder butonuna basabilir.")

print("\n📋 Treeview'da Tıklama Mekanizması:")
print("   1. Kullanıcı 'Mesaj Gönder' sütununa tıklar")
print("   2. on_tree_click event handler tetiklenir")
print("   3. Tıklanan satırdaki telefon numarası alınır")
print("   4. webbrowser.open() ile WhatsApp açılır")
print("   5. 'Mesaj Atıldı Mı?' sütunu 'Evet' olarak güncellenir")

---

## 13. Hata Yönetimi ve Debugging <a id='13-hata-yonetimi'></a>

### Karşılaşılan Hatalar ve Çözümleri

| # | Hata | Sebep | Çözüm |
|---|------|-------|-------|
| 1 | `InvalidSessionIdException` | Chrome yüklü değil | Edge'e geçiş (fallback) |
| 2 | `TimeoutException` (searchboxinput) | Google ID değiştirdi | Çoklu selector sistemi |
| 3 | `RuntimeError: main thread is not in main loop` | Thread'den UI güncellemesi | `_update_ui()` metodu |
| 4 | Çerez ekranı engeli | Google çerez onay sayfası | iframe + buton tıklama |
| 5 | Sonsuz kaydırma döngüsü | Yanlış scroll hedefi | Feed paneli + yükseklik kontrolü |

In [ ]:
# === HATA YÖNETİMİ ÖRNEKLERİ ===

print("🔧 Hata Yönetimi Stratejileri")
print("=" * 50)

strategies = [
    {
        "hata": "Element bulunamadı",
        "strateji": "try/except + 'Bilgi bulunamadı' varsayılan değer",
        "kod": """try:
    business_name = driver.find_element(By.CLASS_NAME, 'DUwDvf').text
except:
    business_name = 'Bilgi bulunamadı'"""
    },
    {
        "hata": "Tarayıcı başlatılamadı",
        "strateji": "Edge → Chrome fallback zinciri",
        "kod": """try:
    driver = webdriver.Edge(options=options)
except:
    driver = webdriver.Chrome(options=options)"""
    },
    {
        "hata": "Thread güvenliği",
        "strateji": "_update_ui() ile güvenli çağrı",
        "kod": """def _update_ui(self, func):
    try:
        self.root.after(0, func)
    except RuntimeError:
        pass"""
    }
]

for i, s in enumerate(strategies, 1):
    print(f"\n{'─' * 50}")
    print(f"  #{i} Hata: {s['hata']}")
    print(f"  Strateji: {s['strateji']}")
    print(f"  Kod:")
    for line in s['kod'].split('\n'):
        print(f"    {line}")

---

## 14. Tam Kaynak Kod <a id='14-tam-kod'></a>

Uygulamanın tam kaynak kodu `isletme_verileri.py` dosyasında bulunmaktadır.

### Çalıştırma

```bash
python isletme_verileri.py
```

### Sonuç

![Uygulama Ekran Görüntüsü](public/Ekran%20görüntüsü%202026-06-06%20210542.png)

---

## 🎓 Öğrenilen Kavramlar

Bu projeyi tamamlayarak şu konularda bilgi edindiniz:

- ✅ **Selenium WebDriver** ile tarayıcı otomasyonu
- ✅ **Tkinter** ile GUI geliştirme
- ✅ **Threading** ile arka plan işlemleri
- ✅ **Pandas** ile veri işleme ve Excel aktarımı
- ✅ **Regular Expressions** ile metin temizleme
- ✅ **CSS Selector ve XPath** ile web element bulma
- ✅ **Anti-bot** korumasını bypass etme teknikleri
- ✅ **Thread-safe** programlama pratikleri